In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import cv2
import numpy as np
import matplotlib.pyplot as plt
import load
from PIL.ImageEnhance import Brightness
from compute_thickness import thickness_batch, brightness_batch
from FM_funcs import OTFlowMatching
import FM_funcs
import configs

import torch.nn.functional as F
# import PGFM_funcs
from PGFMv2_funcs import PGFM
import tqdm
import evaluate
from fid.fid_compute import compute_fid_from_batches

In [ ]:
2

In [ ]:
FM_class = OTFlowMatching()
PGFM_class = PGFM()
mnist_dataset = load.myMMIST()

In [ ]:
train_set = mnist_dataset.get_data(plot_hist=True).to(configs.device)
MNIST_distribution_evaluator = evaluate.distribution_evaluator(train_set)

In [ ]:
train_set = mnist_dataset.get_data_brightness(plot_hist=True).to(configs.device)
MNIST_distribution_evaluator = evaluate.distribution_evaluator(train_set)

In [ ]:
def eval(distribution_evaluator, data_set = train_set, batch_size = 1000, rep_num = 10, type = "brightness"):
    num_record_FM = np.zeros(rep_num)
    num_record_PGFM = np.zeros(rep_num)
    prob_record_FM = np.zeros(rep_num)
    prob_record_PGFM = np.zeros(rep_num)
    swd_record_FM = np.zeros(rep_num)
    swd_record_PGFM = np.zeros(rep_num)

    for i in tqdm.tqdm(range(rep_num)):
        FM_res = FM_funcs.sampler(stage1model, batch_size, stoptime=1, default_generation_step = 100)
        PGFM_res = PGFM_class.RLFMsample( stage1model, stage2model, batch_size, default_generation_step= 100,
                   default_stage1t =configs.default_stage1_t, default_RLstep_S = configs.default_RL_Steps_S)

        if type == "brightness":
            FM_brightness_vec = brightness_batch(FM_res.cpu())
            PGFM_brightness_vec = brightness_batch(PGFM_res.cpu())
            ind_FM,num_FM = configs.count_valid_num_bright(FM_brightness_vec)
            ind_PGFM, num_PGFM = configs.count_valid_num_bright(PGFM_brightness_vec)
            # print(num_PGFM)
        elif type == "thickness":
            FM_thickness_vec = thickness_batch(FM_res.cpu())
            PGFM_thickness_vec = thickness_batch(PGFM_res.cpu())
            ind_FM, num_FM = configs.count_valid_num_thick(FM_thickness_vec)
            ind_PGFM, num_PGFM = configs.count_valid_num_thick(PGFM_thickness_vec)
        else:
            raise NotImplementedError

        prob_record_FM[i] = num_FM / batch_size
        prob_record_PGFM[i] = num_PGFM / batch_size
        num_record_FM[i] = num_FM
        num_record_PGFM[i] = num_PGFM

        ref = PGFM_class.get_samples(data_set, batch_size)
        swd_FM = distribution_evaluator.SWD_after_PCA(ref, FM_res)
        swd_PGFM = distribution_evaluator.SWD_after_PCA(ref, FM_res)

        swd_record_FM[i] = swd_FM
        swd_record_PGFM[i] = swd_PGFM

    print("FM prob mean:", np.mean(prob_record_FM), "std:", np.std(prob_record_FM))
    print("PGFM mean:", np.mean(prob_record_PGFM), "std:", np.std(prob_record_PGFM))
    print("FM num mean:", np.mean(num_record_FM), "std:", np.std(num_record_FM))
    print("PGFM num mean:", np.mean(num_record_PGFM), "std:", np.std(num_record_PGFM))
    print("FM SWD mean:", np.mean(swd_record_FM), "std:", np.std(swd_record_FM))
    print("PGFM SWD mean:", np.mean(swd_record_PGFM), "std:", np.std(swd_record_PGFM))





### Thickness

In [ ]:
ckpt1 = torch.load('./saved_model/FM_MNIST_iter_200000_23.pth', map_location=configs.device, weights_only=True)
# FM_MNIST_bright_iter_200000: bright
# FM_MNIST_iter_200000_23: thick

ckpt2 = torch.load('./saved_model/Apr15RLFM_MNIST_thick_06s20_iter_train2_100000.pth', map_location=configs.device, weights_only=True)
# Apr17RLFM_MNIST_thick_00s60_iter_train2_100000
# Apr17RLFM_MNIST_thick_06s20_iter_train2_60000
#RLFM_MNIST_thick_iter_train2_Mar26
# RLFM_MNIST_thick_06s40_iter_train2_95000
stage1model = FM_class.get_untrained_model()
stage1model.load_state_dict(ckpt1)
stage2model = PGFM_class.policy
stage2model.load_state_dict(ckpt2)

In [ ]:
eval(MNIST_distribution_evaluator, type = "thickness", rep_num = 100)

In [ ]:
1000 - 766.43

In [ ]:
res = FM_funcs.sampler(stage1model, 30000, stoptime=1, default_generation_step = 100)

In [ ]:
# res = train_set
thickness_vec = thickness_batch(res.cpu())
unique_vals, counts = np.unique(thickness_vec, return_counts=True)


plt.figure(figsize=(8, 5))
plt.bar(unique_vals, counts, width=0.1, edgecolor='black')
plt.xlabel("Thickness Value")
plt.ylabel("Frequency")
plt.grid(axis='y', linestyle='--', alpha=0.7)
ind,num = configs.count_valid_num_thick(thickness_vec)
plt.title("Valid num: " + str(num))
plt.xlim(0.9,4.5)
plt.savefig("./figs/thickness_x.png", dpi=300)

plt.show()

In [ ]:
ref = PGFM_class.get_samples(train_set, 21405)

ref = F.interpolate(torch.clip(ref,0,1), size=(32, 32), mode='bilinear', align_corners=False)
res = F.interpolate(torch.clip(res,0,1), size=(32, 32), mode='bilinear', align_corners=False)
# res = torch.rand_like(res)
compute_fid_from_batches(res, ref)

In [ ]:
res = PGFM_class.RLFMsample( stage1model, stage2model, 10000, default_generation_step= 60,
                   default_stage1t =0.6, default_RLstep_S = 40)

In [ ]:
thickness_vec = thickness_batch(res.cpu())
unique_vals, counts = np.unique(thickness_vec, return_counts=True)

plt.figure(figsize=(8, 5))
plt.bar(unique_vals, counts, width=0.1, edgecolor='black')
plt.xlabel("Thickness Value")
plt.ylabel("Frequency")
plt.grid(axis='y', linestyle='--', alpha=0.7)
ind,num = configs.count_valid_num_thick(thickness_vec)
plt.title("Valid num: " + str(num))
plt.xlim(0.9,4.5)
plt.savefig("./figs/thickness_PGFM.png", dpi=300)

plt.show()

In [ ]:
# res = xstage2_N1TD
rand_ind = torch.randperm(len(res))
display_num = 5
plt.figure(figsize=(12,6))
for i in range(display_num):
    plt.subplot(3, display_num, i+1)


    # image_np1 = np.clip(res[rand_ind[i]].cpu().numpy()[0],0,1 )
    image_np1 = res[rand_ind[i]].cpu().numpy()[0]
    plt.imshow(image_np1, cmap='gray')
    plt.axis('off')

    image_np0 = (np.clip(res[rand_ind[i]].cpu().numpy()[0],0,1) * 255).astype(np.uint8)
    # image_np0 = (res[rand_ind[i]].cpu().numpy()[0] * 255).astype(np.uint8)
    image_np = cv2.threshold(image_np0, 128, 255, cv2.THRESH_BINARY)[1]
    plt.subplot(3, display_num, i+6)
    plt.imshow(image_np, cmap='gray')
    plt.grid(False)
    plt.axis('off')
    plt.title(f"{thickness_vec[rand_ind[i]]:.3f}")


plt.show()

### Brightness

In [ ]:
ckpt1 = torch.load('./saved_model/FM_MNIST_bright_iter_200000.pth', map_location=configs.device, weights_only=True)
# FM_MNIST_bright_iter_200000: bright
# FM_MNIST_iter_200000_23: thick

ckpt2 = torch.load('./saved_model/Apr25RLFM_MNIST_bright_06s20_iter_train2_20000.pth', map_location=configs.device, weights_only=True)
# RLFM_MNIST_bright_iter_train2_Mar26
#RLFM_MNIST_bright_06s20_iter_train2_Mar28
stage1model = FM_class.get_untrained_model()
stage1model.load_state_dict(ckpt1)
stage2model = PGFM_class.policy
stage2model.load_state_dict(ckpt2)

In [ ]:
eval(MNIST_distribution_evaluator, type = "brightness", rep_num = 100)

In [ ]:
ref = PGFM_class.get_samples(train_set, 30379)

ref = F.interpolate(ref, size=(32, 32), mode='bilinear', align_corners=False)
res = F.interpolate(res, size=(32, 32), mode='bilinear', align_corners=False)

compute_fid_from_batches(res, ref)

In [ ]:
res = FM_funcs.sampler(stage1model, 10000, stoptime=1, default_generation_step = 100)

In [ ]:
# res = train_set
brightness_vec = brightness_batch(res.cpu())
interval_mid = np.linspace(60, 250, 20)

# for interval in interval_mid:
bin_edges = interval_mid

plt.figure(figsize=(8, 5))
plt.hist(brightness_vec, bins=bin_edges, edgecolor='black', align='mid')
plt.xlabel("brightness Value")
plt.ylabel("Frequency")
plt.grid(axis='y', linestyle='--', alpha=0.7)
ind,num = configs.count_valid_num_bright(brightness_vec)
plt.title("Valid num: " + str(num))
plt.xlim(60, 250)
plt.savefig("./figs/brightness_x.png", dpi=300)

plt.show()

In [ ]:

res = PGFM_class.RLFMsample(stage1model, stage2model, 10000, default_generation_step=100,
                            default_stage1t=0.6, default_RLstep_S=40)
# res = PGFM_class.RLFMsample( stage1model, stage2model, 1000, default_generation_step= 100,
#                    default_stage1t =configs.default_stage1_t, default_RLstep_S = configs.default_RL_Steps_S)

In [ ]:
brightness_vec = brightness_batch(res.cpu())
interval_mid = np.linspace(60, 250, 20)

# for interval in interval_mid:
bin_edges = interval_mid

plt.figure(figsize=(8, 5))
plt.hist(brightness_vec, bins=bin_edges, edgecolor='black', align='mid')
plt.xlabel("brightness Value")
plt.ylabel("Frequency")
plt.grid(axis='y', linestyle='--', alpha=0.7)
ind,num = configs.count_valid_num_bright(brightness_vec)
plt.title("Valid num: " + str(num))
plt.xlim(60, 250)
plt.savefig("./figs/brightness_PGFM.png", dpi=300)

plt.show()

In [ ]:
# res = xstage2_N1TD
rand_ind = torch.randperm(len(res))
display_num = 5
plt.figure(figsize=(12,6))
for i in range(display_num):
    plt.subplot(3, display_num, i+1)


    # image_np1 = np.clip(res[rand_ind[i]].cpu().numpy()[0],0,1 )
    image_np1 = res[rand_ind[i]].cpu().numpy()[0]
    plt.imshow(image_np1, cmap='gray')
    plt.axis('off')

    image_np = (np.clip(res[rand_ind[i]].cpu().numpy()[0],0,1) * 255).astype(np.uint8)
    # image_np = (res[rand_ind[i]].cpu().numpy()[0] * 255).astype(np.uint8)
    image_np = cv2.threshold(image_np, 128, 255, cv2.THRESH_BINARY)[1]
    plt.subplot(3, display_num, i+6)
    plt.imshow(image_np, cmap='gray')
    plt.grid(False)
    plt.axis('off')
    plt.title(f"{brightness_vec[rand_ind[i]]:.3f}")


plt.show()